# 04 · SGD, Adam, AdamW and learning-rate schedules

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/nn-fundamentals/04_optimizers_schedules.ipynb) &nbsp; 📖 Concept deck: [Optimizers](https://nilesh-salpe.github.io/visual-ai/nn-fundamentals/05-optimizers/) &nbsp; ⏱️ Suggested time: **35 min**

**You will build:** SGD with momentum and AdamW as optimizer step functions, the difference between L2 regularisation and decoupled weight decay inside Adam, and a warmup + cosine learning-rate schedule.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


r0 = np.random.default_rng(0)
Xc = r0.normal(size=(400, 10)); w_true = r0.normal(size=10)
yc = (Xc @ w_true + 0.5 * r0.normal(size=400) > 0).astype(float)
def logreg_grad(w):
    p = 1 / (1 + np.exp(-(Xc @ w)))
    return Xc.T @ (p - yc) / len(yc), -np.mean(yc * np.log(p + 1e-12) + (1 - yc) * np.log(1 - p + 1e-12))

## Task 1: Warmup + cosine schedule

Return the learning rate at step `t` (0-indexed): linear warmup from 0 to `peak` over `warmup` steps (lr = peak·(t+1)/warmup), then cosine decay from `peak` to `peak·final_frac` at step `total`.
This is the shape used by many LLM runs (Llama: 2,000 warmup steps, cosine to 10% of the peak).

In [ ]:
def warmup_cosine(t, peak, warmup, total, final_frac=0.1):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_warmup_cosine(f):
    assert np.isclose(f(0, 1.0, 10, 100), 0.1) and np.isclose(f(9, 1.0, 10, 100), 1.0)
    assert np.isclose(f(100, 1.0, 10, 100), 0.1), "ends at final_frac × peak"
    mid = f(55, 1.0, 10, 100)
    assert np.isclose(mid, 0.1 + 0.9 * 0.5), "halfway through the decay the cosine is at its midpoint"
    lrs = [f(t, 1.0, 10, 100) for t in range(10, 101)]
    assert all(a >= b - 1e-12 for a, b in zip(lrs, lrs[1:])), "must decrease after warmup"

check("warmup_cosine", _test_warmup_cosine, warmup_cosine)

## Task 2: One AdamW step

Given parameters `w`, gradient `g`, state `(m, v, t)` and hyper-parameters, return `(w_new, (m, v, t))`.
Adam moments and bias correction as usual; then **decoupled** weight decay (Loshchilov & Hutter): $w \leftarrow w - \eta\,(\hat m/(\sqrt{\hat v} + \epsilon) + \lambda w)$. The decay term does **not** go through m and v.

In [ ]:
def adamw_step(w, g, state, lr, b1=0.9, b2=0.95, eps=1e-8, wd=0.1):
    m, v, t = state
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_adamw_step(f):
    w = np.array([1.0, -2.0]); st = (np.zeros(2), np.zeros(2), 0)
    w1, st1 = f(w, np.zeros(2), st, lr=0.1, wd=0.1)
    assert np.allclose(w1, w * (1 - 0.1 * 0.1)), "with zero gradient, AdamW still shrinks w by lr·wd"
    w2, st2 = f(w, np.array([0.5, 0.5]), st, lr=0.1, wd=0.0)
    assert np.allclose(w2, w - 0.1), "first Adam step moves each coordinate by ≈ lr (bias correction)"
    assert st2[2] == 1

check("adamw_step", _test_adamw_step, adamw_step)

## Task 3: L2 inside Adam is not weight decay

Train logistic regression for 300 steps with Adam (lr 0.05) two ways: (a) **L2**: add λw to the gradient before Adam; (b) **AdamW**: decoupled decay λ. Use λ = 0.1 and start from w = 3·ones.
Return `(norm_l2, norm_adamw)`: the final weight norms. Adam normalises the L2 term away for coordinates with large gradients, so the two differ.

In [ ]:
def l2_vs_decoupled(lam=0.1, steps=300, lr=0.05):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_l2_vs_decoupled(f):
    a, b = f()
    assert not np.isclose(a, b, rtol=0.02), "the two regularisers should give different weights"
    assert a > 0 and b > 0

check("l2_vs_decoupled", _test_l2_vs_decoupled, l2_vs_decoupled)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Warmup + cosine schedule
def warmup_cosine_solution(t, peak, warmup, total, final_frac=0.1):
    if t < warmup:
        return peak * (t + 1) / warmup
    prog = min(1.0, (t - warmup) / max(1, total - warmup))
    return peak * (final_frac + (1 - final_frac) * 0.5 * (1 + np.cos(np.pi * prog)))

check("warmup_cosine (reference)", _test_warmup_cosine, warmup_cosine_solution)

In [ ]:
# @title Solution for Task 2: One AdamW step
def adamw_step_solution(w, g, state, lr, b1=0.9, b2=0.95, eps=1e-8, wd=0.1):
    m, v, t = state; t += 1
    m = b1 * m + (1 - b1) * g; v = b2 * v + (1 - b2) * g * g
    mh, vh = m / (1 - b1 ** t), v / (1 - b2 ** t)
    return w - lr * (mh / (np.sqrt(vh) + eps) + wd * w), (m, v, t)

check("adamw_step (reference)", _test_adamw_step, adamw_step_solution)

In [ ]:
# @title Solution for Task 3: L2 inside Adam is not weight decay
def l2_vs_decoupled_solution(lam=0.1, steps=300, lr=0.05):
    out = []
    for mode in ['l2', 'adamw']:
        w = 3 * np.ones(10); st = (np.zeros(10), np.zeros(10), 0)
        for _ in range(steps):
            g, _ = logreg_grad(w)
            if mode == 'l2':
                w, st = adamw_step_solution(w, g + lam * w, st, lr, wd=0.0)
            else:
                w, st = adamw_step_solution(w, g, st, lr, wd=lam)
        out.append(np.linalg.norm(w))
    return tuple(out)

check("l2_vs_decoupled (reference)", _test_l2_vs_decoupled, l2_vs_decoupled_solution)

---
## Experiments
**What the schedule looks like, and how far the two regularisers pull the weights.** Print a Llama-style schedule (peak 3e-4, 2,000 warmup steps, cosine to 10%) at a few steps, then compare final weight norms for L2-in-Adam vs AdamW across λ.

In [ ]:
for t in [0, 999, 1999, 50_000, 100_000, 199_999]:
    print(f"step {t:7d}: lr = {warmup_cosine_solution(t, 3e-4, 2000, 200_000):.2e}")
for lam in [0.01, 0.1, 0.5]:
    a, b = l2_vs_decoupled_solution(lam)
    print(f"lambda {lam:4}: |w| with L2-in-Adam = {a:.3f}   with AdamW = {b:.3f}")

---
## Takeaways
* AdamW = Adam + weight decay applied directly to the weights, not through the gradient statistics.
* L2 regularisation inside Adam gets divided by √v̂, so heavily-updated weights are barely regularised.
* LLM runs warm the learning rate up, then decay it (often cosine) to a fraction of the peak.